# 31 -- Do the conditioning attributes actually vary?

At k = 3 the dataset takes `sorted(t*.tif)[:3]`, so only t0, t1 and t2
reach the model. If orbit direction and relative orbit are the same across
those three views, two of the three real attributes are constants and carry
no information -- which would explain the +0.031 (p = 0.28) null in
Section 4.

CPU only. No GPU, no model, no sampling. Runs in seconds.


In [1]:
# Orbit-direction diversity in the attributes the model actually sees.
# At k=3 the dataset takes sorted(t*.tif)[:3], i.e. t0, t1, t2 only.

import json
from pathlib import Path
from collections import Counter

WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
CONTEXT_K = 3

patch_dirs = sorted(S1_DIR.glob('s1_patch_*'))
print(f'patches found: {len(patch_dirs)}')

per_view = [Counter() for _ in range(7)]      # what each t index carries
mixed_k = 0                                    # patches whose first k views differ
seen_combo = Counter()                         # the (dir, rel_orbit) set per patch
missing = 0

for d in patch_dirs:
    p = d / 'attrs.json'
    if not p.exists():
        missing += 1
        continue
    attrs = json.load(open(p))
    for i, a in enumerate(attrs[:7]):
        per_view[i][(a.get('acquisition_date'),
                     a.get('orbit_direction'),
                     a.get('relative_orbit_number'))] += 1
    first_k = attrs[:CONTEXT_K]
    dirs = {a.get('orbit_direction') for a in first_k}
    if len(dirs) > 1:
        mixed_k += 1
    seen_combo[tuple(sorted(
        (a.get('orbit_direction'), a.get('relative_orbit_number')) for a in first_k
    ))] += 1

print(f'patches with no attrs.json: {missing}\n')

print('Per view index (date, orbit direction, relative orbit) -> patch count')
for i, c in enumerate(per_view):
    if not c:
        continue
    tag = 'USED' if i < CONTEXT_K else '    '
    for combo, n in c.most_common():
        print(f'  {tag} t{i}: {combo}  x{n}')

print(f'\nPatches whose first {CONTEXT_K} views mix ASCENDING and DESCENDING: '
      f'{mixed_k} / {len(patch_dirs) - missing}')

print(f'\nDistinct (direction, relative orbit) sets across the first {CONTEXT_K} views:')
for combo, n in seen_combo.most_common():
    print(f'  {combo}  x{n}')


patches found: 1676
patches with no attrs.json: 0

Per view index (date, orbit direction, relative orbit) -> patch count
  USED t0: ('2024-03-18', 'ASCENDING', 108)  x1676
  USED t1: ('2024-03-20', 'ASCENDING', 137)  x1676
  USED t2: ('2024-04-01', 'ASCENDING', 137)  x1676
       t3: ('2024-04-13', 'ASCENDING', 137)  x1676
       t4: ('2024-04-23', 'ASCENDING', 108)  x1676
       t5: ('2024-04-25', 'ASCENDING', 137)  x1676
       t6: ('2024-05-07', 'ASCENDING', 137)  x1676

Patches whose first 3 views mix ASCENDING and DESCENDING: 0 / 1676

Distinct (direction, relative orbit) sets across the first 3 views:
  (('ASCENDING', 108), ('ASCENDING', 137), ('ASCENDING', 137))  x1676
